# 03 Ollama Local Provider Integration (OpenClaw, 2026)

## What This Lesson Is
Configure and validate local Ollama usage through OpenClaw with explicit base URL and model policy.

## Scientific Lens
- Concept: Local-provider integration with environment-aware endpoint normalization.
- Measure: Successful configuration and query completion rate.
- Validity Limit: Local model quality/throughput depends on host hardware and model size.


## How It Works
1. Normalize base URLs and build deterministic provider payload safely.
2. Validate selected local model against an approved small-model set.
3. Apply OpenClaw Ollama provider config live and run a minimal query.


In [ ]:
import os
import shutil

HAS_OPENCLAW = shutil.which("openclaw") is not None
print("openclaw available:", HAS_OPENCLAW)
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("OLLAMA_BASE_URL:", os.getenv("OLLAMA_BASE_URL", "<unset>"))


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: runs real OpenClaw CLI operations when available; otherwise prints explicit skip guidance.


In [ ]:
# Deterministic Demo
import json
allowed_models = {"qwen2.5-coder:1.5b", "qwen2.5-coder:7b"}
ollama_base = "http://host.docker.internal:11434".rstrip("/") + "/v1"
selected_model = "qwen2.5-coder:1.5b"
provider_payload = {"baseUrl": ollama_base, "apiKey": "ollama-local", "models": [selected_model]}
print(json.dumps(provider_payload, indent=2))
assert selected_model in allowed_models
assert provider_payload["baseUrl"].endswith("/v1")


In [ ]:
# Live Demo
import json
import os
import shutil
import subprocess

if shutil.which("openclaw") is None:
    print("Skipping live Ollama demo: openclaw CLI is not installed.")
elif not os.getenv("OLLAMA_BASE_URL"):
    print("Skipping live Ollama demo: OLLAMA_BASE_URL is not set.")
else:
    base = os.getenv("OLLAMA_BASE_URL", "").rstrip("/")
    if not base.endswith("/v1"):
        base = f"{base}/v1"
    model = os.getenv("OLLAMA_MODEL", "qwen2.5-coder:1.5b")
    provider = json.dumps({"baseUrl": base, "apiKey": "ollama-local", "models": [model]})
    cmds = [
        ["openclaw", "config", "set", "models.providers.ollama", provider, "--json"],
        ["openclaw", "config", "set", "agents.defaults.model.primary", f"ollama/{model}"],
        ["openclaw", "agent", "--local", "--to", "+15555550123", "--message", "Explain one benefit of local models in one sentence.", "--timeout", "180"],
    ]
    for cmd in cmds:
        print("$", " ".join(cmd))
        proc = subprocess.run(cmd, capture_output=True, text=True)
        print((proc.stdout or proc.stderr).strip()[:1200])


## Applied Labs
1. Parameterize allowed models and reject unsupported models before `config set`.
2. Compare latency between `qwen2.5-coder:1.5b` and one larger local model.
3. Add probe of Ollama `/api/tags` before running OpenClaw query.

## Validation Checklist
- Base URL normalization yields `/v1` endpoint.
- Live flow skips cleanly when required env vars are missing.
- Local model selection is constrained by explicit allowed set.

## Further Reading
- Ollama API: https://github.com/ollama/ollama/blob/main/docs/api.md
- OpenClaw docs: https://docs.openclaw.ai
- Docker networking: https://docs.docker.com/desktop/features/networking/
